In [6]:
# ClickHouse Connection Setup
from clickhouse_driver import Client
import pandas as pd
import configparser
from pathlib import Path

# Load configuration from config file
config = configparser.ConfigParser()
config_path = Path('/root/research-dir/dev/jazzcash-fraud-detection/config/clickhouse_config.ini')

if config_path.exists():
    config.read(config_path)
    CLICKHOUSE_CONFIG = {
        'host': config['clickhouse']['host'],
        'port': int(config['clickhouse']['port']),
        'database': config['clickhouse']['database'],
        'user': config['clickhouse']['user'],
        'password': config['clickhouse']['password']
    }
    print("✅ Configuration loaded from clickhouse_config.ini")
else:
    # Fallback to hardcoded config
    CLICKHOUSE_CONFIG = {
        'host': 'localhost',
        'port': 9000,
        'database': 'public',
        'user': 'default',
        'password': 'DfsTeChB1'
    }
    print("⚠️  Config file not found, using default configuration")

print("\n🔌 Connecting to ClickHouse...")
print(f"📍 Host: {CLICKHOUSE_CONFIG['host']}:{CLICKHOUSE_CONFIG['port']}")
print(f"🗄️  Database: {CLICKHOUSE_CONFIG['database']}")

# Create ClickHouse client
try:
    clickhouse_client = Client(
        host=CLICKHOUSE_CONFIG['host'],
        port=CLICKHOUSE_CONFIG['port'],
        database=CLICKHOUSE_CONFIG['database'],
        user=CLICKHOUSE_CONFIG['user'],
        password=CLICKHOUSE_CONFIG['password'],
        settings={
            'max_execution_time': 7200,  # 2 hour timeout for batch processing
            'send_timeout': 600,
            'receive_timeout': 600,
            'connect_timeout': 10
        }
    )
    
    # Test connection
    result = clickhouse_client.execute('SELECT version()')
    clickhouse_version = result[0][0]
    
    print(f"\n✅ ClickHouse connection successful!")
    print(f"📦 ClickHouse Version: {clickhouse_version}")
    
    # Show available databases
    databases = clickhouse_client.execute('SHOW DATABASES')
    print(f"🗂️  Available Databases: {[db[0] for db in databases]}")
    
    # Show tables in current database
    tables = clickhouse_client.execute(f'SHOW TABLES FROM {CLICKHOUSE_CONFIG["database"]}')
    if tables:
        table_names = [tbl[0] for tbl in tables]
        print(f"📊 Tables in '{CLICKHOUSE_CONFIG['database']}': {len(table_names)} tables found")
        
        # Check for required tables
        required_tables = ['stixor_iar_distributed', 'ac_from_features_distributed']
        for req_table in required_tables:
            if req_table in table_names:
                print(f"   ✓ {req_table}")
            else:
                print(f"   ✗ {req_table} (not found)")
    else:
        print(f"📊 No tables found in '{CLICKHOUSE_CONFIG['database']}'")
    
except Exception as e:
    print(f"\n❌ Failed to connect to ClickHouse: {str(e)}")
    print("\n💡 Troubleshooting Tips:")
    print("   • Ensure ClickHouse server is running")
    print("   • Check if port 9000 is accessible")
    print("   • Verify credentials and permissions")
    print("   • Check config file at: /root/research-dir/dev/jazzcash-fraud-detection/config/clickhouse_config.ini")
    raise

print("\n" + "="*80)

✅ Configuration loaded from clickhouse_config.ini

🔌 Connecting to ClickHouse...
📍 Host: localhost:9000
🗄️  Database: public

✅ ClickHouse connection successful!
📦 ClickHouse Version: 25.10.1.3796
🗂️  Available Databases: ['INFORMATION_SCHEMA', 'default', 'information_schema', 'public', 'system']
📊 Tables in 'public': 21 tables found
   ✓ stixor_iar_distributed
   ✓ ac_from_features_distributed



In [7]:
def execute_clickhouse_query(query, return_df=False):
    """
    Execute a ClickHouse query and optionally return results as DataFrame
    
    Args:
        query: SQL query string
        return_df: If True, return results as pandas DataFrame
    
    Returns:
        Query results or None
    """
    try:
        print(f"🔄 Executing query...")
        result = clickhouse_client.execute(query, with_column_types=True)
        
        if return_df and result:
            # Extract data and column info
            data = result[0] if isinstance(result, tuple) else result
            
            if isinstance(result, tuple) and len(result) > 1:
                # Has column type information
                columns = [col[0] for col in result[1]]
                df = pd.DataFrame(data, columns=columns)
            else:
                df = pd.DataFrame(data)
            
            print(f"✅ Query executed successfully! Rows: {len(df):,}")
            return df
        else:
            print(f"✅ Query executed successfully!")
            return result
            
    except Exception as e:
        print(f"❌ Query execution failed: {str(e)}")
        raise

In [ ]:
# Load fraud datasets
import pandas as pd
fraud_mbar = pd.read_csv('/root/research-dir/fraud_mbar.csv')
fraud_iar = pd.read_csv('/root/research-dir/fraud_iar.csv')

print("✅ Fraud datasets loaded successfully!")
print(f"📊 fraud_mbar shape: {fraud_mbar.shape}")
print(f"📊 fraud_iar shape: {fraud_iar.shape}")

print("\n📋 fraud_mbar columns:")
print(fraud_mbar.columns.tolist())

print("\n📋 fraud_iar columns:")
print(fraud_iar.columns.tolist())

✅ Fraud datasets loaded successfully!
📊 fraud_mbar shape: (1, 24)
📊 fraud_iar shape: (1, 20)

📋 fraud_mbar columns:
['a_c_reference', 'region', 'city', 'registered_channel', 'registered_date_time', 'a_c_status', 'a_c_level', 'agent_group', 'limit_group', 'charge_profile', 'credit_dl_ml_yl', 'debit_dl_ml_yl', 'year_of_birth', 'last_modified_date_time', 'dormant_date', 're_active_date', 'place_of_birth', 'account_type_name', 'mpin_status', 'filer', 'prov', 'year_mdob', 'gmsisdn', 'trust_level']

📋 fraud_iar columns:
['data_date', 'trans_id', 'trans_initiate_time', 'customer_msisdn', 'trx_channel', 'trx_type', 'trx_status', 'ac_from', 'ac_to', 'start_balance', 'trx_amt', 'end_balance', 'utility_company', 'bill_ref_number', 'fee', 'fed', 'reason_type', 'pur_of_remit', 'ec', 'merchant_id']


In [4]:
from datetime import datetime, timedelta
import time
from datetime import date

# Generate all dates in January 2025 (January 1 to January 31)
# Generate all dates from 2025-01-01 to 2025-04-30 (inclusive)
dates_to_process = []
# start = date(2025, 3, 31)
# end = date(2025, 4, 30)  # April has 30 days

# delta = end - start
# for i in range(delta.days + 1):
#     d = start + timedelta(days=i)
#     dates_to_process.append(d.strftime('%Y-%m-%d'))
dates_to_process.append('2025-03-31')
print(f"📅 Total dates to process: {len(dates_to_process)}")
print(f"First date: {dates_to_process[0]}")
print(f"Last date: {dates_to_process[-1]}")

LOOKBACK_DAYS = 7
total_start_time = time.time()
successful_dates = []
failed_dates = []

for idx, CUTOFF_DATE in enumerate(dates_to_process, 1):
    print(f"\n{'='*80}")
    print(f"Processing {idx}/{len(dates_to_process)}: {CUTOFF_DATE}")
    print(f"{'='*80}")
    
    try:
        # Calculate start date
        cutoff = datetime.strptime(CUTOFF_DATE, '%Y-%m-%d')
        start_date = (cutoff - timedelta(days=LOOKBACK_DAYS - 1)).strftime('%Y-%m-%d')
        
        # Build parameterized feature engineering query
        feature_query = f"""
INSERT INTO public.ac_from_features_distributed
WITH 
-- Get users who transacted on cutoff date
active_users AS (
    SELECT DISTINCT ac_from
    FROM public.stixor_iar_distributed
    WHERE data_date = toDate('{CUTOFF_DATE}')
      AND ac_from != ''),
-- Pre-calculate top channels and types per user ({LOOKBACK_DAYS}-day window)
user_channel_stats AS (
    SELECT 
        ac_from,
        trx_channel,
        count() as channel_count,
        row_number() OVER (PARTITION BY ac_from ORDER BY count() DESC) as channel_rank
    FROM (
        SELECT 
            ac_from,
            trx_channel
        FROM public.stixor_iar_distributed
        WHERE data_date >= toDate('{start_date}')
          AND data_date <= toDate('{CUTOFF_DATE}')
          AND ac_from GLOBAL IN (SELECT ac_from FROM active_users)
    )
    GROUP BY ac_from, trx_channel
),
user_type_stats AS (
    SELECT 
        ac_from,
        trx_type,
        count() as type_count,
        row_number() OVER (PARTITION BY ac_from ORDER BY count() DESC) as type_rank
    FROM (
        SELECT 
            ac_from,
            trx_type
        FROM public.stixor_iar_distributed
        WHERE data_date >= toDate('{start_date}')
          AND data_date <= toDate('{CUTOFF_DATE}')
          AND ac_from GLOBAL IN (SELECT ac_from FROM active_users)
    )
    GROUP BY ac_from, trx_type
),
user_recipient_stats AS (
    SELECT 
        ac_from,
        ac_to,
        count() as recipient_count,
        sum(start_balance) as total_to_recipient,
        row_number() OVER (PARTITION BY ac_from ORDER BY count() DESC) as recipient_rank
    FROM (
        SELECT 
            ac_from,
            ac_to,
            start_balance
        FROM public.stixor_iar_distributed
        WHERE data_date >= toDate('{start_date}')
          AND data_date <= toDate('{CUTOFF_DATE}')
          AND ac_from GLOBAL IN (SELECT ac_from FROM active_users)
          AND ac_to != ''
    )
    GROUP BY ac_from, ac_to
)

SELECT 
    main.ac_from,
    toDate('{CUTOFF_DATE}') as cutoff_date,
    
    -- 3-day features (excludes cutoff date, uses days_back 1-3)
    sumIf(1, main.days_back BETWEEN 1 AND 3) as total_txns_3d,
    sumIf(main.start_balance, main.days_back BETWEEN 1 AND 3) as total_amount_3d,
    avgIf(main.start_balance, main.days_back BETWEEN 1 AND 3) as avg_amount_3d,
    quantileIf(0.5)(main.start_balance, main.days_back BETWEEN 1 AND 3) as median_amount_3d,
    maxIf(main.start_balance, main.days_back BETWEEN 1 AND 3) as max_amount_3d,
    minIf(main.start_balance, main.days_back BETWEEN 1 AND 3) as min_amount_3d,
    uniqIf(main.ac_to, main.days_back BETWEEN 1 AND 3) as unique_recipients_3d,
    uniqIf(main.trx_channel, main.days_back BETWEEN 1 AND 3) as unique_channels_3d,
    uniqIf(main.trx_type, main.days_back BETWEEN 1 AND 3) as unique_types_3d,
    
    -- 7-day features (excludes cutoff date, uses days_back 1-7)
    sumIf(1, main.days_back BETWEEN 1 AND 7) as total_txns_7d,
    sumIf(main.start_balance, main.days_back BETWEEN 1 AND 7) as total_amount_7d,
    avgIf(main.start_balance, main.days_back BETWEEN 1 AND 7) as avg_amount_7d,
    quantileIf(0.5)(main.start_balance, main.days_back BETWEEN 1 AND 7) as median_amount_7d,
    maxIf(main.start_balance, main.days_back BETWEEN 1 AND 7) as max_amount_7d,
    minIf(main.start_balance, main.days_back BETWEEN 1 AND 7) as min_amount_7d,
    uniqIf(main.ac_to, main.days_back BETWEEN 1 AND 7) as unique_recipients_7d,
    uniqIf(main.trx_channel, main.days_back BETWEEN 1 AND 7) as unique_channels_7d,
    uniqIf(main.trx_type, main.days_back BETWEEN 1 AND 7) as unique_types_7d,
    
    -- Channel features (7-day for most_used, last_used, diversity)
    anyIf(ch.trx_channel, ch.channel_rank = 1) as most_used_channel_7d,
    argMax(main.trx_channel, main.trans_initiate_time) as last_used_channel,
    if(uniq(main.trx_channel) > 1, 
       1 - (max(ch.channel_count) / sum(ch.channel_count)), 0) as channel_diversity_score_7d,
    
    -- Type features (7-day for most_used, last_used, diversity)
    anyIf(ty.trx_type, ty.type_rank = 1) as most_used_type_7d,
    argMax(main.trx_type, main.trans_initiate_time) as last_used_type,
    if(uniq(main.trx_type) > 1, 
       1 - (max(ty.type_count) / sum(ty.type_count)), 0) as type_diversity_score_7d,
    
    -- Time-based features (7-day, excluding cutoff date)
    sumIf(1, toHour(main.trans_initiate_time) IN (2,3,4,5,6) AND main.days_back BETWEEN 1 AND 7) as night_txns_7d,
    sumIf(1, toDayOfWeek(main.trans_initiate_time) IN (6,7) AND main.days_back BETWEEN 1 AND 7) as weekend_txns_7d,
    sumIf(1, toHour(main.trans_initiate_time) BETWEEN 9 AND 17 AND main.days_back BETWEEN 1 AND 7) as peak_hour_txns_7d,
    sumIf(1, toHour(main.trans_initiate_time) NOT BETWEEN 9 AND 17 AND main.days_back BETWEEN 1 AND 7) as off_peak_hour_txns_7d,
    
    -- Balance features (7-day, excluding cutoff date)
    avgIf(main.start_balance, main.days_back BETWEEN 1 AND 7) as avg_start_balance_7d,
    avgIf(main.end_balance, main.days_back BETWEEN 1 AND 7) as avg_end_balance_7d,
    minIf(least(main.start_balance, main.end_balance), main.days_back BETWEEN 1 AND 7) as min_balance_7d,
    maxIf(greatest(main.start_balance, main.end_balance), main.days_back BETWEEN 1 AND 7) as max_balance_7d,
    stddevPopIf(main.start_balance, main.days_back BETWEEN 1 AND 7) as balance_volatility_7d,
    
    -- Recipient features (7-day, excluding cutoff date)
    anyIf(rs.ac_to, rs.recipient_rank = 1) as top_recipient_7d,
    avgIf(main.start_balance, main.days_back BETWEEN 1 AND 7 AND main.ac_to != '') as avg_amount_per_recipient_7d,
    maxIf(main.start_balance, main.days_back BETWEEN 1 AND 7 AND main.ac_to != '') as max_amount_to_single_recipient_7d,
    if(count(main.ac_from) > 0, max(rs.recipient_count) / count(main.ac_from), 0) as recipient_concentration_ratio_7d,
    
    -- Behavioral features (7-day)
    if(count(main.ac_from) > 1,
       dateDiff('hour', min(main.trans_initiate_time), max(main.trans_initiate_time)) / (count(main.ac_from) - 1),
       0) as avg_time_between_txns_7d,
    count(main.ac_from) / greatest(dateDiff('day', min(main.data_date), max(main.data_date)) + 1, 1) as txn_frequency_score_7d,
    min(main.trans_initiate_time) as first_txn_time,
    max(main.trans_initiate_time) as last_txn_time,
    dateDiff('day', max(main.data_date), toDate('{CUTOFF_DATE}')) as days_since_last_txn,
    
    now() as processing_timestamp,
    toDate(now()) as created_at

FROM (
    SELECT 
        ac_from,
        ac_to,
        trans_id,
        start_balance,
        end_balance,
        trx_channel,
        trx_type,
        trans_initiate_time,
        data_date,
        dateDiff('day', data_date, toDate('{CUTOFF_DATE}')) as days_back
    FROM public.stixor_iar_distributed
    WHERE data_date >= toDate('{start_date}')
      AND data_date <= toDate('{CUTOFF_DATE}')
      AND ac_from GLOBAL IN (SELECT ac_from FROM active_users)
    ORDER BY ac_from, trans_initiate_time
) main
GLOBAL LEFT JOIN user_channel_stats ch ON main.ac_from = ch.ac_from AND main.trx_channel = ch.trx_channel
GLOBAL LEFT JOIN user_type_stats ty ON main.ac_from = ty.ac_from AND main.trx_type = ty.trx_type  
GLOBAL LEFT JOIN user_recipient_stats rs ON main.ac_from = rs.ac_from AND main.ac_to = rs.ac_to
GROUP BY main.ac_from
"""
        
        start_time = time.time()
        
        # Execute the INSERT query
        clickhouse_client.execute(feature_query)
        
        elapsed_time = time.time() - start_time
        
        print(f"✅ Feature engineering completed successfully!")
        print(f"⏱️  Execution time: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")
        
        successful_dates.append(CUTOFF_DATE)
        
    except Exception as e:
        elapsed_time = time.time() - start_time
        print(f"❌ Feature engineering failed after {elapsed_time:.2f} seconds")
        print(f"   Error: {str(e)}")
        failed_dates.append((CUTOFF_DATE, str(e)))

# Summary
total_elapsed_time = time.time() - total_start_time
print(f"\n{'='*80}")
print(f"BATCH PROCESSING COMPLETE")
print(f"{'='*80}")
print(f"⏱️  Total execution time: {total_elapsed_time:.2f} seconds ({total_elapsed_time/60:.2f} minutes)")
print(f"✅ Successful: {len(successful_dates)}/{len(dates_to_process)}")
print(f"❌ Failed: {len(failed_dates)}/{len(dates_to_process)}")

if failed_dates:
    print(f"\nFailed dates:")
    for date, error in failed_dates:
        print(f"   - {date}: {error[:100]}...")
        
# Get final count
try:
    count_query = "SELECT count(*) FROM public.ac_from_features_distributed"
    total_users = clickhouse_client.execute(count_query)[0][0]
    print(f"\n📊 Total records in table: {total_users:,}")
except Exception as e:
    print(f"\n⚠️  Could not retrieve final count: {str(e)}")

Error on localhost:9000 ping: Unexpected EOF while reading bytes
Connection was closed, reconnecting.
Error on socket shutdown: [Errno 107] Transport endpoint is not connected


📅 Total dates to process: 1
First date: 2025-03-31
Last date: 2025-03-31

Processing 1/1: 2025-03-31
✅ Feature engineering completed successfully!
⏱️  Execution time: 44.09 seconds (0.73 minutes)

BATCH PROCESSING COMPLETE
⏱️  Total execution time: 44.12 seconds (0.74 minutes)
✅ Successful: 1/1
❌ Failed: 0/1

📊 Total records in table: 779,680,651


In [19]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def create_features_from_dataframe(df, lookback_days=7):
    """
    Create features from pandas DataFrame without cutoff date filtering
    
    Args:
        df: DataFrame with transaction data
        lookback_days: Number of days to look back (default 7)
    
    Returns:
        DataFrame with features for each ac_from
    """
    
    # Ensure data_date and trans_initiate_time are datetime
    df['data_date'] = pd.to_datetime(df['data_date'])
    df['trans_initiate_time'] = pd.to_datetime(df['trans_initiate_time'])
    
    # Get unique users from all data
    active_users = df['ac_from'].unique()
    
    # Use all available data for feature creation
    window_data = df.copy()
    
    # Calculate days_back relative to the latest transaction date
    latest_date = df['data_date'].max()
    window_data['days_back'] = (latest_date - window_data['data_date']).dt.days
    
    # Initialize results list
    results = []
    
    for ac_from in active_users:
        user_data = window_data[window_data['ac_from'] == ac_from].copy()
        
        if len(user_data) == 0:
            continue
            
        # Helper function for conditional aggregation
        def agg_if(series, condition, agg_func):
            filtered = series[condition]
            if len(filtered) == 0:
                return None
            return agg_func(filtered)
        
        # Create feature dictionary
        features = {
            'ac_from': ac_from,
        }
        
        # 3-day features (last 3 days of available data)
        mask_3d = user_data['days_back'] <= 3
        features.update({
            'total_txns_3d': mask_3d.sum(),
            'total_amount_3d': agg_if(user_data['start_balance'], mask_3d, np.sum),
            'avg_amount_3d': agg_if(user_data['start_balance'], mask_3d, np.mean),
            'median_amount_3d': agg_if(user_data['start_balance'], mask_3d, np.median),
            'max_amount_3d': agg_if(user_data['start_balance'], mask_3d, np.max),
            'min_amount_3d': agg_if(user_data['start_balance'], mask_3d, np.min),
            'unique_recipients_3d': user_data[mask_3d]['ac_to'].nunique(),
            'unique_channels_3d': user_data[mask_3d]['trx_channel'].nunique(),
            'unique_types_3d': user_data[mask_3d]['trx_type'].nunique(),
        })
        
        # 7-day features (last 7 days of available data)
        mask_7d = user_data['days_back'] <= 7
        features.update({
            'total_txns_7d': mask_7d.sum(),
            'total_amount_7d': agg_if(user_data['start_balance'], mask_7d, np.sum),
            'avg_amount_7d': agg_if(user_data['start_balance'], mask_7d, np.mean),
            'median_amount_7d': agg_if(user_data['start_balance'], mask_7d, np.median),
            'max_amount_7d': agg_if(user_data['start_balance'], mask_7d, np.max),
            'min_amount_7d': agg_if(user_data['start_balance'], mask_7d, np.min),
            'unique_recipients_7d': user_data[mask_7d]['ac_to'].nunique(),
            'unique_channels_7d': user_data[mask_7d]['trx_channel'].nunique(),
            'unique_types_7d': user_data[mask_7d]['trx_type'].nunique(),
        })
        
        # Channel features
        user_7d = user_data[mask_7d]
        if len(user_7d) > 0:
            channel_counts = user_7d['trx_channel'].value_counts()
            features['most_used_channel_7d'] = channel_counts.index[0] if len(channel_counts) > 0 else None
            features['last_used_channel'] = user_data.loc[user_data['trans_initiate_time'].idxmax(), 'trx_channel']
            
            if len(channel_counts) > 1:
                features['channel_diversity_score_7d'] = 1 - (channel_counts.iloc[0] / channel_counts.sum())
            else:
                features['channel_diversity_score_7d'] = 0
        else:
            features.update({
                'most_used_channel_7d': None,
                'last_used_channel': None,
                'channel_diversity_score_7d': 0
            })
        
        # Type features
        if len(user_7d) > 0:
            type_counts = user_7d['trx_type'].value_counts()
            features['most_used_type_7d'] = type_counts.index[0] if len(type_counts) > 0 else None
            features['last_used_type'] = user_data.loc[user_data['trans_initiate_time'].idxmax(), 'trx_type']
            
            if len(type_counts) > 1:
                features['type_diversity_score_7d'] = 1 - (type_counts.iloc[0] / type_counts.sum())
            else:
                features['type_diversity_score_7d'] = 0
        else:
            features.update({
                'most_used_type_7d': None,
                'last_used_type': None,
                'type_diversity_score_7d': 0
            })
        
        # Time-based features
        if len(user_7d) > 0:
            user_7d_time = user_7d.copy()
            user_7d_time['hour'] = user_7d_time['trans_initiate_time'].dt.hour
            user_7d_time['dow'] = user_7d_time['trans_initiate_time'].dt.dayofweek
            
            features.update({
                'night_txns_7d': ((user_7d_time['hour'] >= 2) & (user_7d_time['hour'] <= 6)).sum(),
                'weekend_txns_7d': (user_7d_time['dow'].isin([5, 6])).sum(),  # Saturday=5, Sunday=6
                'peak_hour_txns_7d': ((user_7d_time['hour'] >= 9) & (user_7d_time['hour'] <= 17)).sum(),
                'off_peak_hour_txns_7d': ((user_7d_time['hour'] < 9) | (user_7d_time['hour'] > 17)).sum(),
            })
        else:
            features.update({
                'night_txns_7d': 0,
                'weekend_txns_7d': 0,
                'peak_hour_txns_7d': 0,
                'off_peak_hour_txns_7d': 0,
            })
        
        # Balance features
        if len(user_7d) > 0:
            features.update({
                'avg_start_balance_7d': user_7d['start_balance'].mean(),
                'avg_end_balance_7d': user_7d['end_balance'].mean(),
                'min_balance_7d': np.minimum(user_7d['start_balance'], user_7d['end_balance']).min(),
                'max_balance_7d': np.maximum(user_7d['start_balance'], user_7d['end_balance']).max(),
                'balance_volatility_7d': user_7d['start_balance'].std(),
            })
        else:
            features.update({
                'avg_start_balance_7d': None,
                'avg_end_balance_7d': None,
                'min_balance_7d': None,
                'max_balance_7d': None,
                'balance_volatility_7d': None,
            })
        
        # Recipient features
        user_7d_recipients = user_7d[user_7d['ac_to'] != '']
        if len(user_7d_recipients) > 0:
            recipient_counts = user_7d_recipients['ac_to'].value_counts()
            recipient_amounts = user_7d_recipients.groupby('ac_to')['start_balance'].sum()
            
            features.update({
                'top_recipient_7d': recipient_counts.index[0] if len(recipient_counts) > 0 else None,
                'avg_amount_per_recipient_7d': user_7d_recipients['start_balance'].mean(),
                'max_amount_to_single_recipient_7d': recipient_amounts.max(),
                'recipient_concentration_ratio_7d': recipient_counts.iloc[0] / len(user_7d) if len(recipient_counts) > 0 else 0,
            })
        else:
            features.update({
                'top_recipient_7d': None,
                'avg_amount_per_recipient_7d': None,
                'max_amount_to_single_recipient_7d': None,
                'recipient_concentration_ratio_7d': 0,
            })
        
        # Behavioral features
        if len(user_data) > 1:
            time_diff_hours = (user_data['trans_initiate_time'].max() - user_data['trans_initiate_time'].min()).total_seconds() / 3600
            features['avg_time_between_txns_7d'] = time_diff_hours / (len(user_data) - 1)
        else:
            features['avg_time_between_txns_7d'] = 0
        
        date_range_days = (user_data['data_date'].max() - user_data['data_date'].min()).days + 1
        features['txn_frequency_score_7d'] = len(user_data) / max(date_range_days, 1)
        
        features.update({
            'first_txn_time': user_data['trans_initiate_time'].min(),
            'last_txn_time': user_data['trans_initiate_time'].max(),
            'days_since_last_txn': (latest_date - user_data['data_date'].max()).days,
            'processing_timestamp': datetime.now(),
            'created_at': datetime.now().date(),
        })
        
        results.append(features)
    
    return pd.DataFrame(results)

# Process the existing fraud_iar dataframe
feature_df = create_features_from_dataframe(df=fraud_iar)

print(f"✅ Feature engineering completed successfully!")
print(f"📊 Generated features for {len(feature_df)} users")
print(f"🔧 Feature columns: {len(feature_df.columns)}")
print("\nSample features:")
print(feature_df.head())


✅ Feature engineering completed successfully!
📊 Generated features for 1 users
🔧 Feature columns: 45

Sample features:
                    ac_from  total_txns_3d  total_amount_3d  avg_amount_3d  \
0  4G5mpzxoT9xxhEl3m9gy0g==              1         76000.67       76000.67   

   median_amount_3d  max_amount_3d  min_amount_3d  unique_recipients_3d  \
0          76000.67       76000.67       76000.67                     1   

   unique_channels_3d  unique_types_3d  ...  avg_amount_per_recipient_7d  \
0                   1                1  ...                     76000.67   

   max_amount_to_single_recipient_7d  recipient_concentration_ratio_7d  \
0                           76000.67                               1.0   

   avg_time_between_txns_7d  txn_frequency_score_7d      first_txn_time  \
0                         0                     1.0 2025-09-21 19:26:20   

        last_txn_time  days_since_last_txn       processing_timestamp  \
0 2025-09-21 19:26:20                    0 2025